# 02 Create Frozen Splits and Manifests

This notebook creates the deterministic train/validation/test manifest that all downstream notebooks consume. Thresholds are selected only on the validation split; final metrics are computed only on the test split.


In [ ]:
from pathlib import Path
import random
import re

import numpy as np
import pandas as pd


In [ ]:
SEED = 42
REPO_ROOT = Path('..').resolve()
WINDOWED_ROOT = REPO_ROOT / 'data' / '01_windowed_labeled_2,5s'
SPECTROGRAM_ROOT = REPO_ROOT / 'data' / '02_spectrograms_150x100px_dataset'
MANIFEST_DIR = REPO_ROOT / 'reports' / 'manifests'
MANIFEST_PATH = MANIFEST_DIR / 'turning_split_seed42.csv'
SUMMARY_PATH = MANIFEST_DIR / 'turning_split_summary.csv'

VALIDATION_TEST_FRACTION = 0.5
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def parse_run_metadata(npz_path: Path) -> dict:
    stickout = npz_path.parent.parent.name
    run_name = npz_path.parent.name
    stem = npz_path.stem
    window_match = re.search(r'_window_(\d+)$', stem)
    rpm_match = re.search(r'rpm(\d+)', run_name)
    doc_match = re.search(r'doc([^_]+)', run_name)
    return {
        'source_dataset': 'turning',
        'process': 'turning',
        'stickout': stickout,
        'source_run': f'{stickout}/{run_name}',
        'rpm': int(rpm_match.group(1)) if rpm_match else np.nan,
        'doc': doc_match.group(1) if doc_match else '',
        'window': int(window_match.group(1)) if window_match else np.nan,
        'sample_id': f'{stickout}_{stem}',
    }

def relative_to_repo(path: Path) -> str:
    return path.resolve().relative_to(REPO_ROOT).as_posix()


In [ ]:
image_index = {}
for image_path in SPECTROGRAM_ROOT.rglob('*.png'):
    parts = image_path.relative_to(SPECTROGRAM_ROOT).parts
    if len(parts) < 3:
        continue
    split, label = parts[0], parts[1]
    name = image_path.stem
    window_part = name.split('_window_')[-1]
    npz_stem = name[: name.rfind(f'_{label}')]
    key = (npz_stem, label)
    image_index[key] = (split, image_path)

rows = []
for npz_path in sorted(WINDOWED_ROOT.rglob('*.npz')):
    with np.load(npz_path) as data:
        label = str(data['label'])
    meta = parse_run_metadata(npz_path)
    image_key = (f'{npz_path.parent.parent.name}_{npz_path.stem}', label)
    image_split, image_path = image_index.get(image_key, ('missing', None))
    rows.append({
        **meta,
        'label': label,
        'original_image_split': image_split,
        'npz_path': relative_to_repo(npz_path),
        'image_path': relative_to_repo(image_path) if image_path else '',
    })

manifest = pd.DataFrame(rows)
display(manifest.head())
manifest.groupby(['label', 'original_image_split']).size().rename('n').reset_index()


In [ ]:
rng = random.Random(SEED)
manifest['split'] = ''

train_mask = (manifest['label'] == 'no_chatter') & (manifest['original_image_split'] == 'train')
manifest.loc[train_mask, 'split'] = 'train'

heldout_mask = manifest['split'].eq('')
heldout = manifest.loc[heldout_mask].copy()

assigned_indices = []
for label, label_rows in heldout.groupby('label'):
    groups = list(label_rows.groupby('source_run').groups.items())
    rng.shuffle(groups)
    n_test_groups = max(1, int(round(len(groups) * VALIDATION_TEST_FRACTION))) if groups else 0
    test_group_names = {name for name, _ in groups[:n_test_groups]}
    for source_run, idx in groups:
        split = 'test' if source_run in test_group_names else 'validation'
        manifest.loc[list(idx), 'split'] = split
        assigned_indices.extend(list(idx))

if manifest['split'].eq('').any():
    raise RuntimeError('Some samples were not assigned to a split.')

def expected_image_path(row: pd.Series) -> str:
    npz_path = REPO_ROOT / row['npz_path']
    experiment = npz_path.parent.parent.name
    out_name = experiment + "_" + npz_path.stem + "_" + str(row["label"]) + ".png"
    image_path = SPECTROGRAM_ROOT / row['split'] / row['label'] / out_name
    return relative_to_repo(image_path)

manifest['image_path'] = manifest.apply(expected_image_path, axis=1)
summary = manifest.groupby(['source_dataset', 'split', 'label']).size().rename('n').reset_index()
display(summary)


In [ ]:
manifest.to_csv(MANIFEST_PATH, index=False)
summary.to_csv(SUMMARY_PATH, index=False)
print(f'Wrote {MANIFEST_PATH}')
print(f'Wrote {SUMMARY_PATH}')
